# Silver Layer: Cleaning

**Target Tables:**
- **Read:** `bronze_ad_links` (Bronze Layer)
- **Write:** `silver_clean_ads` (Parquet)

**Objective:**
This notebook is responsible for cleaning and transforming the raw data extracted in the Bronze layer. It applies business rules, handles missing values, normalizes text (e.g., prices, descriptions, and dates), and structures the data into a clean, queryable format suitable for analysis in the Gold layer.

**To-Do:**
- Define strict schema enforcement for the Silver tables.
- Implement string manipulation for currency (BRL) and dates.
- Filter out invalid, duplicated, or outlier advertisements.

In [1]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parents[2])
if project_root not in sys.path:
    sys.path.append(project_root)

from sqlalchemy import insert, select
from sqlalchemy.orm import Session
import polars as pl
import re
import unicodedata

# Importing from 'app' module
from app.config import db_engine
from app.models import GeneralSearch, InformationExtraction

In [2]:
with db_engine.connect() as connection:
    # Table with general instructions
    df_general = (
        pl.read_database(
            select(GeneralSearch)
            .where(
                (GeneralSearch.available == True)
                & (GeneralSearch.status == 200)
            ),
            connection=connection
        )
    )

    # Details' table
    df_details = (
        pl.read_database(
            select(InformationExtraction),
            connection=connection
        )
    )

In [3]:
def clean_col_name(col_name: str) -> str:
    lower_name = col_name.lower()
    
    # Removes special characters
    no_special_chars_name = (
        unicodedata
        .normalize('NFKD', lower_name)
        .encode('ASCII', 'ignore')
        .decode('utf-8')
    )

    # Replace spaces by '_'
    no_spaces_name = (
        no_special_chars_name
        .replace(' ', '_')
    )

    # Removes duplicated underscores
    final_name = re.sub(r'_+', '_', no_spaces_name)

    return final_name

In [ ]:
initial_full_df = (
    df_general
    .join(
        df_details,
        left_on="id",
        right_on="general_search_id",
        how="right" # Discard invalid or out-of-region general scraps
    )
    .unnest('specifications')
)

# Renaming coluns by removing spaces and special characters, also lowering them
col_mapping = {col: clean_col_name(col) for col in initial_full_df.columns}
renamed_initial_full_df = initial_full_df.rename(col_mapping)

In [ ]:
col_normalized_full_df = (
    renamed_initial_full_df
    .with_columns(
        (
            pl.when(
                pl.col('para_doacao') == 'Sim'
            ).then(
                pl.lit(True)
            ).otherwise(
                pl.lit(False)
            )
        ).alias('ForDonation'),
        (
            pl.when(
                pl.col('aceita_trocas') == 'Sim'
            ).then(
                pl.lit(True)
            ).otherwise(
                pl.lit(False)
            )
        ).alias('AcceptTrades'),
        (
            pl.col('caracteristicas')
            .str.replace_all("Inclui ", "")
            .str.split(r', ')
        ).alias('Characteristics')
    )
    .select(
        pl.col('category').alias('Category'),
        pl.col('subcategory').alias('Subcategory'),
        pl.col('item').alias('Item'),
        pl.col('url').alias('Url'),
        pl.col('region').alias('Region'),
        pl.col('link').alias('Link'),
        pl.col('datetime').dt.date().alias('Date'),
        pl.col('store').alias('Store'),
        pl.col('id').alias('Id'),
        pl.col('first_image_src').alias('FirstImageSrc'),
        pl.col('title').str.replace_all(r'\s+', ' ').str.strip_chars().str.to_titlecase().alias('Title'),
        pl.col('description').str.replace_all(r'\s+', ' ').str.strip_chars().str.to_titlecase().alias('Description'),
        pl.col('currency').alias('Currency'),
        pl.col('price').alias('Price'),
        pl.col('marca').str.strip_chars().alias('Brand'),
        pl.col('condicao').str.strip_chars().alias('ItemCondition'),
        pl.col('marca_do_processador').alias('CpuBrand'),
        pl.col('modelo_do_processador').alias('CpuModel'),
        pl.col('memoria_ram').str.extract(r'(\d+)').cast(pl.Int32).alias('RamGb'),
        pl.col('marca_da_placa_de_video').alias('GpuBrand'),
        pl.col('tamanho_de_tela').str.extract(r'(\d+)').cast(pl.Float32).alias('ScreenSizePol'),
        pl.col('armazenamento').str.extract(r'(\d+)').cast(pl.Int32).alias('StorageGb'),
        pl.col('AcceptTrades'),
        pl.col('Characteristics'),
        pl.col('ForDonation')
    )
)

In [18]:
col_normalized_full_df

Category,Subcategory,Item,URL,Region,Link,Date,Store,Id,FirstImageSrc,Title,Description,Currency,Price,Brand,ItemCondition,CpuBrand,CpuModel,RamGb,GpuBrand,ScreenSizePol,StorageGb,AcceptTrades,Characteristics,ForDonation
str,str,str,str,str,str,date,str,i64,str,str,str,str,f64,str,str,str,str,i32,str,f32,i32,bool,list[str],bool
"""notebooks""","""popular""","""Samsung Galaxy Book Go""","""/brasil?q=Samsung+Galaxy+Book+…","""mg""","""https://mg.olx.com.br/regiao-d…",2026-07-16,"""OLX""",1,"""https://img.olx.com.br/images/…","""Samsung Galaxy Book Go , Windo…","""Notebook Samsung Galaxy Book G…","""R$""",2100.0,"""Samsung""","""Usado - Excelente""",null,null,4,"""Outros""",14.0,128,false,"[""Acessórios"", ""Bluetooth"", … ""Wi-fi""]",false
"""notebooks""","""popular""","""Samsung Galaxy Book Go""","""/brasil?q=Samsung+Galaxy+Book+…","""rs""","""https://rs.olx.com.br/regioes-…",2026-07-16,"""OLX""",2,"""https://img.olx.com.br/images/…","""Notebook Samsung Galaxy Book G…","""Notebook Samsung Galaxy Book G…","""R$""",1700.0,"""Samsung""","""Novo""",null,null,4,null,14.0,128,false,null,false
"""notebooks""","""popular""","""Samsung Galaxy Book Go""","""/brasil?q=Samsung+Galaxy+Book+…","""sp""","""https://sp.olx.com.br/regiao-d…",2026-07-16,"""OLX""",3,"""https://img.olx.com.br/images/…","""Samsung Galaxy Book Go Com Pro…","""Comprei Com Milhas Mas Nunca U…","""R$""",2000.0,"""Samsung""","""Usado - Excelente""",null,null,4,"""Outros""",14.0,128,false,"[""Acessórios"", ""Bluetooth"", … ""Wi-fi""]",false
"""notebooks""","""popular""","""Samsung Galaxy Book Go""","""/brasil?q=Samsung+Galaxy+Book+…","""sp""","""https://sp.olx.com.br/sao-paul…",2026-07-16,"""OLX""",4,"""https://img.olx.com.br/images/…","""Notebook Samsung Galaxy Book G…","""Notebook Samsung Galaxy Book G…","""R$""",1900.0,"""Samsung""","""Usado - Excelente""","""Amd""","""Amd Ryzen 9""",64,"""Intel""",12.0,128,false,"[""Acessórios"", ""Bluetooth"", … ""Wi-fi""]",false
"""notebooks""","""popular""","""Samsung Galaxy Book Go""","""/brasil?q=Samsung+Galaxy+Book+…","""sp""","""https://sp.olx.com.br/sao-paul…",2026-07-16,"""OLX""",5,"""https://img.olx.com.br/images/…","""Samsung Galaxy Book Go""","""Eleve Sua Produtividade Com O …","""R$""",900.0,"""Samsung""","""Usado - Excelente""",null,null,4,"""Outros""",14.0,128,false,"[""Bluetooth"", ""Bluetooth"", … ""Wi-fi""]",false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""notebooks""","""popular""","""Acer Aspire 5""","""/brasil?q=Acer+Aspire+5""","""sp""","""https://sp.olx.com.br/vale-do-…",2026-07-16,"""OLX""",96,"""https://img.olx.com.br/images/…","""Notebook Acer Aspire 5 | Video…","""Acer Aspire 5 (A514-53G) - Pot…","""R$""",2300.0,"""Acer""","""Usado - Excelente""","""Intel""","""Intel Core I5""",8,"""Nvidia""",16.0,512,false,null,false
"""notebooks""","""popular""","""Acer Aspire 5""","""/brasil?q=Acer+Aspire+5""","""sp""","""https://sp.olx.com.br/sao-paul…",2026-07-16,"""OLX""",97,"""https://img.olx.com.br/images/…","""Notebook Acer Aspire 5 | Intel…","""Notebook Comprado Em Setembro …","""R$""",2900.0,"""Acer""","""Usado - Excelente""","""Intel""","""Intel Core I5""",16,"""Nvidia""",14.0,512,false,"[""Bluetooth"", ""Cabos"", … ""Wi-fi""]",false
"""notebooks""","""popular""","""Acer Aspire 5""","""/brasil?q=Acer+Aspire+5""","""sp""","""https://sp.olx.com.br/sao-paul…",2026-07-16,"""OLX""",98,"""https://img.olx.com.br/images/…","""Notebook Acer Aspire 5 Usado""","""Notebook Acer Aspire 5, Usado …","""R$""",1290.0,"""Acer""","""Usado - Bom""","""Intel""","""Intel Core I5""",8,"""Intel""",null,256,false,null,false


In [ ]:
if not col_normalized_full_df.is_empty():
    with Session(db_engine) as session:
        # Saving on sqlite
        session.execute(insert(InformationExtraction), col_normalized_full_df.to_dicts())
        session.commit()
else:
    print("No data found to insert.")